### Schiebeparkplatz

[Video](https://youtu.be/Lj0FpBap8hU)

<img src='schiebeparkplatz.png' width='250'>

Hinweis: das Programm muss eine in Bezug auf die Anzahl der zu verschiebenden Autos optimale Lösung finden, die Länge des Verschiebungswegs spielt keine Rolle: Ob ich 1 nach links oder 2 nach rechts gehe, ist gleich gut.

#### Beispieldaten

In [190]:
# Anschauen der Beispieldaten 0-5
nr = '0'
print(f'Beispiel {nr}:')
f = open(f'beispieldaten/parkplatz{nr}.txt')
print(f.read().strip())
f.close()

Beispiel 0:
A G
2
H 2
I 5


Jede Datei beschreibt einen Schiebeparkplatz und enthält

  * in der ersten Zeile die Bezeichnung des ersten und letzten Autos
    auf den normalen Parkplätzen (Bezeichnungen „A“, „B“, „C“, usw.),
  * in der zweiten Zeile die Anzahl n der quer stehenden Autos und
  * in den weiteren n Zeilen für jedes quer stehende Auto jeweils.
    * die Bezeichnung des quer stehenden Autos und
    * die Position $p_i$ des quer stehenden Autos.

Die Position startet links von 0. Jedes verschiebbare Auto nimmt wie in
der Zeichnung gezeigt zwei Positionen ein (also jeweils $p_i$ und $p_{i + 1}$).

#### Einlesen der Daten

In [221]:
nr = 0
print(f'Beispiel {nr}:')

f = open('./beispieldaten/parkplatz'+str(nr)+'.txt')
c1, c2 = f.readline().split()     # die Zeichen für die obere Wagenreihe
n = int(f.readline())             # anzahl querstehender Wagen

quer = []               # Liste mit Namen und Position der querstehenden Wagen
for i in range(n):
    name, pos = f.readline().split()
    quer.append((name,int(pos)))

print(f'Obere Reihe von {c1} bis {c2}.')
print(f'Anzahl querstehender Wagen: {n}')
print(f'Positionen der querstehenden Wagen: {quer}') 

Beispiel 0:
Obere Reihe von A bis G.
Anzahl querstehender Wagen: 2
Positionen der querstehenden Wagen: [('H', 2), ('I', 5)]


#### Schrittweise Lösung

Die obere Reihe als Liste von Buchstaben

In [222]:
oben = [chr(c) for c in range(ord(c1), ord(c2)+1)]    # obere Wagenreihe als Liste mit Buchstaben
oben

['A', 'B', 'C', 'D', 'E', 'F', 'G']

Wir wollen eine Lösung mit der Breitensuche finden. Ein state soll die Situation der querstehenden Autos zeigen. Wir modellieren den state als Tuple. Die einzelnen Positionen des Tuples zeigen, welches Auto dort steht, oder ob da freier Platz ist:

    ('.','.','H','H','.','I','I')

modelliert den Anfangszustand des Beispiels in der Aufgabenstellung.


In [223]:
# Konstruktion des startstates als Tuple
startstate = ['.']*len(oben)
for c,k in quer:
    startstate[k]=c
    startstate[k+1]=c
    
startstate = tuple(startstate)
startstate

('.', '.', 'H', 'H', '.', 'I', 'I')

In [224]:
# Zur Visualisierung der Situation
def show(state):
    print(*oben)
    print(*state)
    
show(startstate)

A B C D E F G
. . H H . I I


Zur Suche in dem Baum der Möglichkeiten benötigen wir die Funktionen *goaltest* und *nextstates*.

In [240]:
raus = 2
def goaltest(state):
    ''' raus = index des Wagens in der oberen Reihe, der rausgefahren werden soll '''
    return state[raus] == '.'

In [241]:
def nextstates(state):
    tmp = []
    for c, pos in quer:
        new_state = list(state)
        k = pos
        while k+2 < len(state) and state[k+2]=='.':   # solange rechts Platz ist
            new_state[k]='.'                          # verschiebe die beiden Zeichen in new_state                     
            new_state[k+2]=c                          # um eins nach rechts
            tmp.append(tuple(new_state))              # speichere den Zustand als möglichen Folgezustand
            k+=1            
        k = pos
        new_state = list(state)
        while k-1 >= 0 and new_state[k-1]=='.':       # solange links Platz ist
            new_state[k-1]=c
            new_state[k+1]='.'            
            tmp.append(tuple(new_state))
            k-=1
    return tmp

In [242]:
nextstates(startstate) 

[('.', '.', '.', 'H', 'H', 'I', 'I'),
 ('.', 'H', 'H', '.', '.', 'I', 'I'),
 ('H', 'H', '.', '.', '.', 'I', 'I'),
 ('.', '.', 'H', 'H', 'I', 'I', '.')]

In [243]:
from collections import deque
def bfs(startstate):
    frontier =  deque([startstate])
    prev = {startstate:None}
    while frontier:
        state = frontier.popleft()  
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            if v not in prev:
                frontier.append(v)
                prev[v] = state
    return None, None

def reconstructPath(prev,goalstate):
    state = goalstate
    path = []
    while state is not None:
        path.append(state)
        state = prev[state]
    path.reverse()
    return path

In [244]:
prev, goal = bfs(startstate)
path = reconstructPath(prev,goal) 
path

[('.', '.', 'H', 'H', '.', 'I', 'I'), ('.', '.', '.', 'H', 'H', 'I', 'I')]

Um die Abfolge des Pfades in Worten wiederzugeben, definieren wir uns eine Funktion *getMove*

In [245]:
def getMoves(path):
    '''
    returns: Beschreibung des Pfads als eine Folge von Aktionen (Moves)
    '''
    moves = ''
    for i in range(len(path)-1):
        moves+=getMove(path[i],path[i+1])
    return moves

def getMove(s1, s2):
    for c, _ in quer:
        i1 = s1.index(c)
        i2 = s2.index(c)
        if i1 > i2:
            return f'{c}:{i1-i2} nach links  ' 
        if i1 < i2:
            return f'{c}:{i2-i1} nach rechts  '     

In [246]:
getMoves(path)

'H:1 nach rechts  '

In einer Schleife ermitteln wir die Ergebnisse für alle Wagen der oberen Reihe.

In [247]:
for i in range(len(oben)):
    raus = i
    prev, state = bfs(startstate)
    path = reconstructPath(prev, state)
    print(f'{oben[i]}: {getMoves(path)}') 

A: 
B: 
C: H:1 nach rechts  
D: H:1 nach links  
E: 
F: H:1 nach links  I:2 nach links  
G: I:1 nach links  


### Das vollständige Programm

In [251]:
from collections import deque
def bfs(startstate):
    frontier =  deque([startstate])
    prev = {startstate:None}
    while frontier:
        state = frontier.popleft()  
        if goaltest(state):
            return prev, state
        for v in nextstates(state):
            if v not in prev:
                frontier.append(v)
                prev[v] = state
    return None, None

def reconstructPath(prev,goalstate):
    state = goalstate
    path = []
    while state is not None:
        path.append(state)
        state = prev[state]
    path.reverse()
    return path

def getMoves(path):
    '''
    returns: Beschreibung des Pfads als eine Folge von Aktionen (Moves)
    '''
    moves = ''
    for i in range(len(path)-1):
        moves+=getMove(path[i],path[i+1])
    return moves

def nextstates(state):
    tmp = []
    for c, pos in quer:
        k = pos
        new_state = list(state)
        while k+2 < len(state) and state[k+2]=='.':   # solange rechts Platz ist
            new_state[k]='.'                          # verschiebe die beiden Zeichen in new_state                     
            new_state[k+2]=c                          # um eins nach rechts
            tmp.append(tuple(new_state))              # speichere den Zustand als möglichen Folgezustand
            k+=1
            
        k = pos                     
        new_state = list(state)
        while k-1 >= 0 and new_state[k-1]=='.':       # solange links Platz ist
            new_state[k-1]=c
            new_state[k+1]='.'
            tmp.append(tuple(new_state))
            k-=1
    return tmp

 
def goaltest(state):
    ''' nutzt Variable raus = index des Wagens in der oberen Reihe, der rausgefahren werden soll '''
    return state[raus] == '.'

def getMove(s1, s2):
    for c, _ in quer:
        i1 = s1.index(c)
        i2 = s2.index(c)
        if i1 > i2:
            return f'{c}:{i1-i2} nach links  ' 
        if i1 < i2:
            return f'{c}:{i2-i1} nach rechts  '  

nr = 2
print(f'Beispiel {nr}:')

f = open('./beispieldaten/parkplatz'+str(nr)+'.txt')
c1, c2 = f.readline().split()     # die Zeichen für die obere Wagenreihe
n = int(f.readline())             # anzahl querstehender Wagen

quer = []               # Liste mit Namen und Position der querstehenden Wagen
for i in range(n):
    name, pos = f.readline().split()
    quer.append((name,int(pos)))

oben = [chr(c) for c in range(ord(c1), ord(c2)+1)]    # obere Wagenreihe als Liste mit Buchstaben
                                  
startstate = ['.']*len(oben)        # startstates als Tuple
for c,k in quer:
    startstate[k]=c
    startstate[k+1]=c
startstate = tuple(startstate)

show(startstate)
print()

for i in range(len(oben)):
    raus = i
    prev, state = bfs(startstate)
    path = reconstructPath(prev, state)
    print(f'{oben[i]}: {getMoves(path)}') 
 

Beispiel 2:
A B C D E F G H I J K L M N
. . O O . P P Q Q R R . S S

A: 
B: 
C: O:1 nach rechts  
D: O:1 nach links  
E: 
F: O:1 nach links  P:2 nach links  
G: P:1 nach links  
H: R:1 nach rechts  Q:1 nach rechts  
I: P:1 nach links  Q:1 nach links  
J: R:1 nach rechts  
K: P:1 nach links  Q:1 nach links  R:1 nach links  
L: 
M: P:1 nach links  Q:1 nach links  R:1 nach links  S:2 nach links  
N: S:1 nach links  
